# Project 1: Loan Approval Classification
This notebook implements and compares two machine learning models for predicting loan approval:

- Decision Tree
- Logistic Regression

In [1]:
# importing libraries
import pandas as pd
import numpy as np


# import train_test_split for dividing the data into training and testing sets
from sklearn.model_selection import train_test_split
# import GridSearchCV for hyperparameter tuning
from sklearn.model_selection import GridSearchCV
# import StandardScaler for scaling features for Logistic Regression
from sklearn.preprocessing import StandardScaler
# import DecisionTreeClassifier for building the Decision Tree model
from sklearn.tree import DecisionTreeClassifier
# import LogisticRegression for building the Logistic Regression model
from sklearn.linear_model import LogisticRegression
# import evaluation metrics for measuring model performance.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score


In [2]:
# load dataset
df = pd.read_csv("loan_data.csv")

# display first 5 rows 
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


## Data Preparation Summary

The dataset was first inspected to understand its size, column names, data types, and target distribution. Missing values were checked and unrealistic age values were removed as they would reduce data quality. After that, categorical variables were encoded, the data was split using a stratified train-test split, and feature scaling is applied for logistic regression.

In [3]:
# print # of rows and columns
print("Dataset shape:", df.shape)

# print all column names
print("\nColumn names:")
print(df.columns)


Dataset shape: (45000, 14)

Column names:
Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='str')


In [4]:
# print info about data types and non-null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [5]:
# print the count of each class in the target variable
print(df["loan_status"].value_counts())

# print the proportion of each class in the target variable
print(df["loan_status"].value_counts(normalize=True))

loan_status
0    35000
1    10000
Name: count, dtype: int64
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64


## Target Distribution

The target variable `loan_status` was inspected using frequency counts and normalized proportions. This step helps determine whether the classes are balanced and confirms the prediction target for the classification models.

In [6]:
# check the number of missing values in each column
print(df.isnull().sum())

# check the total number of missing values in the dataset
print("Total missing values:", df.isnull().sum().sum())

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64
Total missing values: 0


## Missing Values
Missing values were checked using `df.isnull().sum()` and `df.isnull().sum().sum()`. No missing values were found in the dataset, so no imputation / row removal was necessary. Hence, the dataset could move directly to the next preparation step.

In [7]:
# display summary statistics for the age column.
print(df["person_age"].describe())

count    45000.000000
mean        27.764178
std          6.045108
min         20.000000
25%         24.000000
50%         26.000000
75%         30.000000
max        144.000000
Name: person_age, dtype: float64


## Outlier Handling

The dataset was checked for unrealistic values in important numeric fields. During inspection of the `person_age` column, extremely high age values were observed that are not realistic for loan applicants. These records were treated as data quality issues rather than meaningful observations.

To improve data quality, rows with `person_age` greater than 100 were removed. This step was taken because unrealistic ages could distort model learning and reduce the reliability of the results, especially for Logistic Regression.

In [8]:
#sStore original dataset shape before removing unrealistic ages
original_shape = df.shape

# keep only rows where person_age is less than or equal to 100.
df = df[df["person_age"] <= 100]

# print the dataset shape before and after removing unrealistic ages.
print("Original dataset shape:", original_shape)
print("Dataset shape after removing unrealistic ages:", df.shape)

Original dataset shape: (45000, 14)
Dataset shape after removing unrealistic ages: (44993, 14)


In [9]:
# create the feature matrix X by removing the target column
X = df.drop("loan_status", axis=1)

#cCreate the target vector y using the loan_status column.
y = df["loan_status"]

## Encoding

The feature matrix contained categorical variables such as gender, education, home ownership, loan intent, and previous loan default status. These categorical variables were converted into numeric form using one-hot encoding so that the machine learning models could process them correctly.

In [10]:
# convert categorical variables into dummy variables
X = pd.get_dummies(X, drop_first=True)

# display the first five rows of the encoded feature matrix
X.head()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,person_gender_male,person_education_Bachelor,...,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_Yes
0,22.0,71948.0,0,35000.0,16.02,0.49,3.0,561,False,False,...,True,False,False,True,False,False,False,True,False,False
1,21.0,12282.0,0,1000.0,11.14,0.08,2.0,504,False,False,...,False,False,True,False,True,False,False,False,False,True
2,25.0,12438.0,3,5500.0,12.87,0.44,3.0,635,False,False,...,False,False,False,False,False,False,True,False,False,False
3,23.0,79753.0,0,35000.0,15.23,0.44,2.0,675,False,True,...,False,False,False,True,False,False,True,False,False,False
4,24.0,66135.0,1,35000.0,14.27,0.53,4.0,586,True,False,...,True,False,False,True,False,False,True,False,False,False


In [11]:
# split the dataset into training and testing sets using stratification.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# print the shapes of the training and testing sets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (35994, 22)
X_test shape: (8999, 22)
y_train shape: (35994,)
y_test shape: (8999,)


## Train-Test Split

A stratified train-test split was used so that the proportion of the loan approval classes remained similar in both the training and testing sets. This is important in classification problems because it provides a fairer and more reliable model evaluation.

In [12]:
# create a StandardScaler object for feature scaling
scaler = StandardScaler()

# fit the scaler on the training features and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# transform the test features using the same fitted scaler
X_test_scaled = scaler.transform(X_test)

## Feature Scaling

Feature scaling was applied for Logistic Regression because the model is sensitive to differences in feature magnitude. StandardScaler was used to standardize the features. Scaling was not applied to the Decision Tree model because Decision Trees are based on split thresholds and do not depend on feature scale in the same way.

In [13]:
## Decision Tree Model

In [14]:
# create the baseline Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

# train the baseline Decision Tree model using the training data
dt_model.fit(X_train, y_train)

# predict the loan status values for the test set
y_pred_dt = dt_model.predict(X_test)

In [15]:
# calculate and print the baseline Decision Tree accuracy
print("Baseline Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

# print the classification report for the baseline Decision Tree model
print("\nBaseline Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

# print the confusion matrix for the baseline Decision Tree model
print("\nBaseline Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Baseline Decision Tree Accuracy: 0.8943215912879209

Baseline Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.93      6999
           1       0.76      0.78      0.77      2000

    accuracy                           0.89      8999
   macro avg       0.85      0.85      0.85      8999
weighted avg       0.90      0.89      0.89      8999


Baseline Decision Tree Confusion Matrix:
[[6498  501]
 [ 450 1550]]


In [16]:
# define the hyperparameter grid for the Decision Tree model
dt_param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

# create the GridSearchCV object for Decision Tree tuning
dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# git the grid search on the training data.
dt_grid.fit(X_train, y_train)

# print the best Decision Tree hyperparameters found by the grid search
print("Best Decision Tree Parameters:", dt_grid.best_params_)

Best Decision Tree Parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}


In [17]:
# store the best tuned Decision Tree model
best_dt_model = dt_grid.best_estimator_

# predict the loan status values for the test set using the tuned model
y_pred_best_dt = best_dt_model.predict(X_test)

# calculate and print the tuned Decision Tree accuracy
print("Tuned Decision Tree Accuracy:", accuracy_score(y_test, y_pred_best_dt))

# print the classification report for the tuned Decision Tree model
print("\nTuned Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_best_dt))

# print the confusion matrix for the tuned Decision Tree model
print("\nTuned Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_dt))

Tuned Decision Tree Accuracy: 0.9162129125458385

Tuned Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.98      0.95      6999
           1       0.90      0.70      0.79      2000

    accuracy                           0.92      8999
   macro avg       0.91      0.84      0.87      8999
weighted avg       0.92      0.92      0.91      8999


Tuned Decision Tree Confusion Matrix:
[[6844  155]
 [ 599 1401]]


In [18]:
# store the baseline Decision Tree accuracy
baseline_dt_accuracy = accuracy_score(y_test, y_pred_dt)

# store the tuned Decision Tree accuracy
tuned_dt_accuracy = accuracy_score(y_test, y_pred_best_dt)

# print the baseline and tuned Decision Tree accuracies for comparison
print("Baseline Decision Tree Accuracy:", baseline_dt_accuracy)
print("Tuned Decision Tree Accuracy:", tuned_dt_accuracy)

Baseline Decision Tree Accuracy: 0.8943215912879209
Tuned Decision Tree Accuracy: 0.9162129125458385


In [19]:
# create a DataFrame containing feature importance values from the tuned Decision Tree model
dt_feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_dt_model.feature_importances_
})

# sort the features from most important to least important
dt_feature_importance = dt_feature_importance.sort_values(by="Importance", ascending=False)

# display the top 10 most important features
dt_feature_importance.head(10)

,Feature,Importance
21,previous_loan_defaults_on_file_Yes,0.414127
5,loan_percent_income,0.186218
4,loan_int_rate,0.180875
1,person_income,0.092149
15,person_home_ownership_RENT,0.049613
7,credit_score,0.019305
17,loan_intent_HOMEIMPROVEMENT,0.010052
14,person_home_ownership_OWN,0.009080
20,loan_intent_VENTURE,0.007023
18,loan_intent_MEDICAL,0.006914


## Decision Tree Interpretation

The Decision Tree model can be interpreted using feature importance values. Features with higher importance contributed more to the model’s split decisions and therefore had greater influence on loan approval predictions. By reviewing the most important features, it is possible to understand which applicant and loan characteristics were most influential in the tree’s behavior.

## Logistic Regression Model

In [20]:
# create the baseline Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# train the baseline Logistic Regression model using the scaled training data
lr_model.fit(X_train_scaled, y_train)

# predict the loan status values for the scaled test set
y_pred_lr = lr_model.predict(X_test_scaled)

In [21]:
# calculate and print the baseline Logistic Regression accuracy
print("Baseline Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

# print the classification report for the baseline Logistic Regression model
print("\nBaseline Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

# print the confusion matrix for the baseline Logistic Regression model
print("\nBaseline Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Baseline Logistic Regression Accuracy: 0.894766085120569

Baseline Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93      6999
           1       0.77      0.75      0.76      2000

    accuracy                           0.89      8999
   macro avg       0.85      0.84      0.85      8999
weighted avg       0.89      0.89      0.89      8999


Baseline Logistic Regression Confusion Matrix:
[[6557  442]
 [ 505 1495]]


In [22]:
# fefine the hyperparameter grid for Logistic Regression
lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

# create the GridSearchCV object for Logistic Regression tuning
lr_grid = GridSearchCV(
    estimator=LogisticRegression(random_state=42, max_iter=1000),
    param_grid=lr_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# fit the grid search on the scaled training data
lr_grid.fit(X_train_scaled, y_train)

# print the best Logistic Regression hyperparameters found by the grid search
print("Best Logistic Regression Parameters:", lr_grid.best_params_)

Best Logistic Regression Parameters: {'C': 0.1, 'solver': 'liblinear'}


In [23]:
# store the best tuned Logistic Regression model
best_lr_model = lr_grid.best_estimator_

# predict the loan status values for the scaled test set using the tuned model
y_pred_best_lr = best_lr_model.predict(X_test_scaled)

# calculate and print the tuned Logistic Regression accuracy
print("Tuned Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_best_lr))

# print the classification report for the tuned Logistic Regression model
print("\nTuned Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_best_lr))

# print the confusion matrix for the tuned Logistic Regression model
print("\nTuned Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_lr))

Tuned Logistic Regression Accuracy: 0.8945438382042449

Tuned Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93      6999
           1       0.77      0.75      0.76      2000

    accuracy                           0.89      8999
   macro avg       0.85      0.84      0.85      8999
weighted avg       0.89      0.89      0.89      8999


Tuned Logistic Regression Confusion Matrix:
[[6557  442]
 [ 507 1493]]


In [24]:
# create a DataFrame containing Logistic Regression coefficients
lr_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": best_lr_model.coef_[0]
})

# compute the odds ratio for each feature by exponentiating the coefficients
lr_coefficients["Odds_Ratio"] = np.exp(lr_coefficients["Coefficient"])

# compute the absolute value of each coefficient to rank feature influence
lr_coefficients["Absolute_Coefficient"] = lr_coefficients["Coefficient"].abs()

# sort the coefficients by absolute value from largest to smallest
lr_coefficients = lr_coefficients.sort_values(by="Absolute_Coefficient", ascending=False)

# display the top 10 most influential Logistic Regression features
lr_coefficients.head(10)

,Feature,Coefficient,Odds_Ratio,Absolute_Coefficient
21,previous_loan_defaults_on_file_Yes,-3.127073,0.043846,3.127073
5,loan_percent_income,1.374336,3.952452,1.374336
4,loan_int_rate,0.971209,2.641137,0.971209
3,loan_amnt,-0.640619,0.526966,0.640619
20,loan_intent_VENTURE,-0.464363,0.628536,0.464363
7,credit_score,-0.441145,0.643299,0.441145
15,person_home_ownership_RENT,0.375628,1.455905,0.375628
16,loan_intent_EDUCATION,-0.367018,0.692797,0.367018
14,person_home_ownership_OWN,-0.328933,0.719691,0.328933
19,loan_intent_PERSONAL,-0.265555,0.766780,0.265555


## Logistic Regression Interpretation

The Logistic Regression coefficients show the direction and strength of each feature’s relationship with loan approval. A positive coefficient means that as the feature increases, the likelihood of loan approval also increases, while a negative coefficient means the feature decreases the likelihood of approval.

Odds ratios were calculated by exponentiating the coefficients. An odds ratio greater than 1 indicates increased odds of approval, while an odds ratio less than 1 indicates decreased odds of approval. This makes the model easier to interpret in practical financial terms.

## Business Interpretation

From a business perspective, Logistic Regression provides a clear explanation of how different applicant and loan characteristics influence loan approval outcomes. This is especially useful in financial decision-making because it supports transparency and makes it easier to justify decisions. Features with strong positive or negative effects can help identify key approval and risk factors.

## Model Comparison

In [ ]:
# create a comparison table that summarizes the performance of all models.
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Tuned Decision Tree",
        "Baseline Logistic Regression",
        "Tuned Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_best_dt),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_best_lr)
    ],
    "Precision": [
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_best_dt),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_best_lr)
    ],
    "Recall": [
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_best_dt),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_best_lr)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_best_dt),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_best_lr)
    ]
})

# display the model comparison table.
results

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline Decision Tree,0.894322,0.755729,0.7750,0.765243
1,Tuned Decision Tree,0.916213,0.900386,0.7005,0.787964
2,Baseline Logistic Regression,0.894766,0.771812,0.7475,0.759462
3,Tuned Logistic Regression,0.894544,0.771576,0.7465,0.758831
